# ARC-v0.13 — FEVER Boundary External Replication

This notebook is a **cross-dataset external-validity replication** of the ARC-v0.9
HotpotQA boundary/stability analysis.

The design is intentionally frozen before examining FEVER boundary outcomes.

Primary questions:

1. Does the PQ32↔SQ8 approximation-feedback boundary remain heterogeneous on FEVER?
2. Can the same first-order baseline feature family predict high H3 amplification out of sample?
3. Are stable/null and reversal cases retained?
4. How close are FEVER held-out discrimination and regime structure to HotpotQA ARC-v0.9?

This notebook does **not** claim that FEVER reproduces HotpotQA automatically.
A null or weaker replication is retained as a valid result.

TEST remains untouched.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%pip install -q faiss-cpu==1.12.0 psutil pyarrow scikit-learn pandas matplotlib

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import gc
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
import psutil

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    roc_auc_score,
    average_precision_score,
    accuracy_score,
)

print("FAISS:", faiss.__version__)
print("NumPy:", np.__version__)
print("RAM GiB:", psutil.virtual_memory().total / 1024**3)

## 1. Frozen paths and provenance

In [ ]:
SEED = 20260816
DIM = 384
N_DOCS = 5_416_568
NPROBE = 64
TOP_RETRIEVE = 100
TOP_K = 10
MAX_ROUNDS = 4

ROOT = Path(
    "/content/drive/MyDrive/"
    "hc-rars-fever-5m-untouched-confirmation-v1"
)

ARC_ROOT = Path(
    "/content/drive/MyDrive/"
    "rag-pq-checkpoints/arc-v0"
)

INDEX_ROOT = Path(
    "/content/drive/MyDrive/"
    "rag-pq-checkpoints/arc-index-cache"
)

CORPUS_MEMMAP = (
    ROOT / "stage1/corpus_embeddings.float16.memmap"
)

QUERY_EMB = (
    ROOT / "stage1/query_embeddings_v2.float32.npy"
)

QUERY_IDS = (
    ROOT / "stage1/query_ids.utf8.txt"
)

SPLIT_MANIFEST = (
    ROOT / "stage1/official_split_manifest.json"
)

DEV_QRELS = (
    ROOT / "stage2/dev_qrels_rows.csv"
)

PQ32_PATH = (
    INDEX_ROOT
    / "fever5m-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss"
)

SQ8_PATH = (
    INDEX_ROOT
    / "fever5m-bge-small-ivfsq8-nlist4096-seed20260816.faiss"
)

# HotpotQA v0.9 is used only as an external reference after the FEVER protocol is sealed.
HOTPOT_V09_ROOT = (
    ARC_ROOT / "boundary-stability-map-v09"
)

required = {
    "CORPUS_MEMMAP": CORPUS_MEMMAP,
    "QUERY_EMB": QUERY_EMB,
    "QUERY_IDS": QUERY_IDS,
    "SPLIT_MANIFEST": SPLIT_MANIFEST,
    "DEV_QRELS": DEV_QRELS,
    "PQ32_INDEX": PQ32_PATH,
    "SQ8_INDEX": SQ8_PATH,
}

for name, path in required.items():
    print(f"{name:20s}", "OK" if path.is_file() else "MISSING", path)
    if not path.is_file():
        raise FileNotFoundError(path)

print("\nFEVER PREFLIGHT — PASS")

## 2. Seal the FEVER boundary-replication protocol

In [ ]:
def sha256_file(path, chunk_size=64 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

OUT_ROOT = (
    ARC_ROOT / "fever-boundary-external-replication-v013"
)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

MEAN_ALPHAS = [0.1, 0.3, 0.5, 0.7]
MEAN_KS = [5, 20, 50]

SOFTMAX_ALPHAS = [0.1, 0.3, 0.5, 0.7]
SOFTMAX_KS = [5, 20]
SOFTMAX_TEMPS = [0.05, 0.1, 0.2, 0.5]

BOUNDARY_CONFIGS = []

for alpha in MEAN_ALPHAS:
    for k in MEAN_KS:
        BOUNDARY_CONFIGS.append({
            "method": "mean",
            "alpha": alpha,
            "k": k,
            "temperature": None,
        })

for alpha in SOFTMAX_ALPHAS:
    for k in SOFTMAX_KS:
        for temperature in SOFTMAX_TEMPS:
            BOUNDARY_CONFIGS.append({
                "method": "softmax",
                "alpha": alpha,
                "k": k,
                "temperature": temperature,
            })

assert len(BOUNDARY_CONFIGS) == 44

protocol = {
    "status": "FEVER_BOUNDARY_EXTERNAL_REPLICATION_SEALED_BEFORE_SWEEP",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "FEVER",
    "split": "DEV",
    "expected_dev_queries": 6666,
    "encoder": "BAAI/bge-small-en-v1.5",
    "corpus_rows": N_DOCS,
    "dimension": DIM,
    "retriever_pair": ["PQ32", "SQ8"],
    "nprobe": NPROBE,
    "top_retrieve": TOP_RETRIEVE,
    "feedback_rounds": MAX_ROUNDS,
    "query_split": "deterministic SHA256 parity 50/50 fit-validation",
    "boundary_grid_config_count": len(BOUNDARY_CONFIGS),
    "boundary_configs": BOUNDARY_CONFIGS,
    "prediction_target": "H3_slope",
    "high_amplification_definition":
        "FIT-family 75th percentile of H3_slope; threshold frozen before validation",
    "regime_threshold_abs_slope": 0.002,
    "retain_null_and_reversal_cases": True,
    "test_access_allowed": False,
    "hotpotqa_v09_used_for_parameter_tuning": False,
}

raw = json.dumps(
    protocol,
    sort_keys=True,
    separators=(",", ":"),
).encode("utf-8")

protocol_sha = hashlib.sha256(raw).hexdigest()
protocol["protocol_sha256"] = protocol_sha

PROTOCOL_PATH = OUT / "v013_fever_boundary_protocol.json"
PROTOCOL_PATH.write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8",
)

print("Output:", OUT)
print("Protocol SHA-256:", protocol_sha)
print("ARC-v0.13 PROTOCOL — SEALED")

## 3. Load FEVER DEV queries, qrels, and corpus embeddings

In [ ]:
queries = np.load(
    QUERY_EMB,
    mmap_mode="r",
)

with open(QUERY_IDS, "r", encoding="utf-8") as f:
    query_ids = [x.strip() for x in f if x.strip()]

with open(SPLIT_MANIFEST, "r", encoding="utf-8") as f:
    split = json.load(f)

dev_ids = [str(x) for x in split["dev_query_ids"]]

assert len(dev_ids) == 6666

for key in [
    "test_qrels_relevance_values_accessed",
    "test_retrieval_performed",
    "test_outcomes_observed",
]:
    if key in split:
        assert split[key] is False, (key, split[key])

query_row = {
    str(qid): i
    for i, qid in enumerate(query_ids)
}

missing = [q for q in dev_ids if q not in query_row]
assert not missing, missing[:10]

dev_rows = np.asarray(
    [query_row[q] for q in dev_ids],
    dtype=np.int64,
)

Q_DEV = np.asarray(
    queries[dev_rows],
    dtype=np.float32,
)

Q_DEV /= np.maximum(
    np.linalg.norm(Q_DEV, axis=1, keepdims=True),
    1e-12,
)

qrels_df = pd.read_csv(DEV_QRELS)

print("qrels columns:", qrels_df.columns.tolist())

# Flexible schema normalization for the already-generated FEVER row-qrels artifact.
qid_col = next(
    c for c in ["query_id", "query-id", "qid"]
    if c in qrels_df.columns
)

row_col = next(
    c for c in ["corpus_row", "corpus-row", "row_id", "doc_row"]
    if c in qrels_df.columns
)

score_candidates = [
    c for c in ["score", "relevance", "rel"]
    if c in qrels_df.columns
]

qrels_df[qid_col] = qrels_df[qid_col].astype(str)
qrels_df[row_col] = qrels_df[row_col].astype(np.int64)

if score_candidates:
    score_col = score_candidates[0]
    positive = qrels_df[qrels_df[score_col] > 0]
else:
    positive = qrels_df

dev_qrels = {
    str(qid): set(
        g[row_col].astype(np.int64).tolist()
    )
    for qid, g in positive.groupby(qid_col)
}

assert all(q in dev_qrels for q in dev_ids)

CORPUS = np.memmap(
    CORPUS_MEMMAP,
    dtype=np.float16,
    mode="r",
    shape=(N_DOCS, DIM),
)

print("Q_DEV:", Q_DEV.shape)
print("Corpus:", CORPUS.shape)
print("DEV qrels:", len(dev_qrels))
print("FEVER DEV ALIGNMENT — PASS")

## 4. Deterministic 50/50 boundary fit / validation split

In [ ]:
def split_bucket(qid):
    h = hashlib.sha256(
        str(qid).encode("utf-8")
    ).digest()

    return int.from_bytes(
        h[:8],
        "big",
    ) % 2

boundary_split = pd.DataFrame({
    "query_id": dev_ids,
    "split": [
        "fit"
        if split_bucket(qid) == 0
        else "validation"
        for qid in dev_ids
    ],
})

boundary_split.to_csv(
    OUT / "v013_boundary_query_split.csv",
    index=False,
)

fit_mask = (
    boundary_split["split"].to_numpy()
    == "fit"
)

val_mask = ~fit_mask

print(boundary_split["split"].value_counts())
assert fit_mask.sum() + val_mask.sum() == 6666
print("FEVER BOUNDARY SPLIT — FROZEN")

## 5. Retrieval, feedback, metric, and slope helpers

In [ ]:
def cfg_key(cfg):
    method = cfg["method"]
    k = int(cfg["k"])
    alpha = str(cfg["alpha"]).replace(".", "p")

    if cfg["temperature"] is None:
        temp = "none"
    else:
        temp = str(cfg["temperature"]).replace(".", "p")

    return f"{method}-k{k}-a{alpha}-t{temp}"

def cosine_distance_rows(a, b):
    an = a / np.maximum(
        np.linalg.norm(a, axis=1, keepdims=True),
        1e-12,
    )
    bn = b / np.maximum(
        np.linalg.norm(b, axis=1, keepdims=True),
        1e-12,
    )
    return (
        1.0
        - np.sum(an * bn, axis=1)
    ).astype(np.float32)

def jaccard_distance_rows(a, b):
    out = np.empty(len(a), np.float32)

    for i in range(len(a)):
        A = set(map(int, a[i]))
        B = set(map(int, b[i]))
        union = len(A | B)
        out[i] = (
            1.0
            - len(A & B) / max(union, 1)
        )

    return out

def score_entropy(scores, k=20):
    z = np.asarray(
        scores[:, :k],
        np.float64,
    )

    z -= z.max(
        axis=1,
        keepdims=True,
    )

    p = np.exp(
        np.clip(z, -60, 60)
    )

    p /= np.maximum(
        p.sum(axis=1, keepdims=True),
        1e-12,
    )

    return -np.sum(
        p * np.log(np.maximum(p, 1e-12)),
        axis=1,
    )

def score_margin(scores):
    return np.asarray(
        scores[:, 0] - scores[:, 9],
        np.float32,
    )

def ndcg_at_10(qids, ids):
    discounts = (
        1.0
        / np.log2(np.arange(2, 12))
    )

    out = np.zeros(
        len(qids),
        np.float32,
    )

    for i, qid in enumerate(qids):
        rel = dev_qrels[str(qid)]

        hits = np.asarray(
            [
                int(doc) in rel
                for doc in ids[i, :10]
            ],
            dtype=np.float64,
        )

        dcg = float(
            (hits * discounts).sum()
        )

        ideal = min(len(rel), 10)

        idcg = float(
            discounts[:ideal].sum()
        )

        out[i] = (
            dcg / idcg
            if idcg
            else 0.0
        )

    return out

def fetch_doc_vectors(ids):
    flat = np.asarray(ids, np.int64).reshape(-1)

    if (flat < 0).any():
        raise ValueError("FAISS returned negative IDs")

    vec = np.asarray(
        CORPUS[flat],
        dtype=np.float32,
    )

    vec /= np.maximum(
        np.linalg.norm(vec, axis=1, keepdims=True),
        1e-12,
    )

    return vec.reshape(
        ids.shape[0],
        ids.shape[1],
        DIM,
    )

def feedback_matrix(ids, scores, cfg):
    k = int(cfg["k"])

    ids_k = ids[:, :k]
    scores_k = np.asarray(
        scores[:, :k],
        np.float64,
    )

    docs = fetch_doc_vectors(ids_k)

    if cfg["method"] == "mean":
        weights = np.full(
            (len(ids_k), k),
            1.0 / k,
            dtype=np.float64,
        )

    elif cfg["method"] == "softmax":
        T = float(cfg["temperature"])

        z = scores_k / T
        z -= z.max(axis=1, keepdims=True)

        weights = np.exp(
            np.clip(z, -60, 60)
        )

        weights /= np.maximum(
            weights.sum(axis=1, keepdims=True),
            1e-12,
        )

    else:
        raise ValueError(cfg["method"])

    fb = np.sum(
        docs * weights[:, :, None],
        axis=1,
    ).astype(np.float32)

    fb /= np.maximum(
        np.linalg.norm(fb, axis=1, keepdims=True),
        1e-12,
    )

    return fb

def anchored_update(q0, feedback, alpha):
    q = (
        (1.0 - float(alpha)) * q0
        + float(alpha) * feedback
    ).astype(np.float32)

    q /= np.maximum(
        np.linalg.norm(q, axis=1, keepdims=True),
        1e-12,
    )

    return q

def slopes_from_trajectory(df):
    rows = []

    group_cols = [
        "query_id",
        "low",
        "high",
        "method",
        "alpha",
        "k",
        "temperature",
        "config_key",
    ]

    for keys, g in df.groupby(
        group_cols,
        dropna=False,
        sort=False,
    ):
        (
            qid,
            low,
            high,
            method,
            alpha,
            k,
            temp,
            cfgk,
        ) = keys

        g = g.sort_values("iteration")
        x = g["iteration"].to_numpy(np.float64)

        def slope(col):
            return float(
                np.polyfit(
                    x,
                    g[col].to_numpy(np.float64),
                    1,
                )[0]
            )

        rows.append({
            "query_id": str(qid),
            "low": low,
            "high": high,
            "method": method,
            "alpha": float(alpha),
            "k": int(k),
            "temperature": temp,
            "config_key": cfgk,
            "H1_slope": slope("query_divergence"),
            "H2_slope": slope("candidate_increment"),
            "H3_slope": slope("abs_utility_gap"),
        })

    return pd.DataFrame(rows)

## 6. Baseline PQ32 / SQ8 retrieval and initial-state features

In [ ]:
faiss.omp_set_num_threads(8)

baseline_cache = {}

for name, path in {
    "PQ32": PQ32_PATH,
    "SQ8": SQ8_PATH,
}.items():

    print("\n" + "=" * 80)
    print(name)

    index = faiss.read_index(str(path))
    index.nprobe = NPROBE

    t0 = time.perf_counter()

    scores, ids = index.search(
        np.ascontiguousarray(
            Q_DEV,
            np.float32,
        ),
        TOP_RETRIEVE,
    )

    dt = time.perf_counter() - t0

    baseline_cache[name] = {
        "scores": scores,
        "ids": ids,
        "ndcg": ndcg_at_10(dev_ids, ids),
        "entropy20": score_entropy(scores, 20),
        "margin1_10": score_margin(scores),
    }

    print("seconds :", dt)
    print(
        "nDCG@10:",
        float(
            baseline_cache[name]["ndcg"].mean()
        ),
    )

    del index
    gc.collect()

pq = baseline_cache["PQ32"]
sq = baseline_cache["SQ8"]

features_df = pd.DataFrame({
    "query_id": dev_ids,
    "initial_candidate_divergence":
        jaccard_distance_rows(
            pq["ids"],
            sq["ids"],
        ),
    "initial_abs_utility_gap":
        np.abs(
            sq["ndcg"] - pq["ndcg"]
        ),
    "pq32_entropy20":
        pq["entropy20"],
    "sq8_entropy20":
        sq["entropy20"],
    "pq32_margin1_10":
        pq["margin1_10"],
    "sq8_margin1_10":
        sq["margin1_10"],
    "entropy_gap":
        np.abs(
            pq["entropy20"] - sq["entropy20"]
        ),
    "margin_gap":
        np.abs(
            pq["margin1_10"] - sq["margin1_10"]
        ),
})

assert len(features_df) == 6666
assert features_df.isna().sum().sum() == 0

features_df.to_parquet(
    OUT / "v013_initial_query_features.parquet",
    index=False,
)

print("BASELINE FEATURE RECONSTRUCTION — PASS")
display(features_df.head())

## 7. Synchronized PQ32↔SQ8 trajectory runner

In [ ]:
def run_pair_trajectory(cfg, query_mask):
    idx = np.flatnonzero(query_mask)

    qids = [
        dev_ids[i]
        for i in idx
    ]

    q0 = Q_DEV[idx].copy()

    low = faiss.read_index(str(PQ32_PATH))
    high = faiss.read_index(str(SQ8_PATH))

    low.nprobe = NPROBE
    high.nprobe = NPROBE

    qL = q0.copy()
    qH = q0.copy()

    frames = []
    base_cdiv = None

    for t in range(MAX_ROUNDS + 1):
        sL, idL = low.search(
            np.ascontiguousarray(
                qL,
                np.float32,
            ),
            TOP_RETRIEVE,
        )

        sH, idH = high.search(
            np.ascontiguousarray(
                qH,
                np.float32,
            ),
            TOP_RETRIEVE,
        )

        qdiv = cosine_distance_rows(
            qL,
            qH,
        )

        cdiv = jaccard_distance_rows(
            idL,
            idH,
        )

        if t == 0:
            base_cdiv = cdiv.copy()

        nL = ndcg_at_10(qids, idL)
        nH = ndcg_at_10(qids, idH)

        frames.append(
            pd.DataFrame({
                "query_id": qids,
                "iteration": t,
                "query_divergence": qdiv,
                "candidate_increment":
                    cdiv - base_cdiv,
                "abs_utility_gap":
                    np.abs(nH - nL),
            })
        )

        if t < MAX_ROUNDS:
            qL = anchored_update(
                q0,
                feedback_matrix(
                    idL,
                    sL,
                    cfg,
                ),
                cfg["alpha"],
            )

            qH = anchored_update(
                q0,
                feedback_matrix(
                    idH,
                    sH,
                    cfg,
                ),
                cfg["alpha"],
            )

    del low, high
    gc.collect()

    out_df = pd.concat(
        frames,
        ignore_index=True,
    )

    out_df["low"] = "PQ32"
    out_df["high"] = "SQ8"
    out_df["method"] = cfg["method"]
    out_df["alpha"] = cfg["alpha"]
    out_df["k"] = cfg["k"]
    out_df["temperature"] = cfg["temperature"]
    out_df["config_key"] = cfg_key(cfg)

    return out_df

## 8. FIT sweep

This is the expensive stage.

The notebook writes one parquet per configuration, so interrupted Colab runs can resume
without recomputing completed configurations.

In [ ]:
fit_frames = []

for i, cfg in enumerate(
    BOUNDARY_CONFIGS,
    1,
):
    key = cfg_key(cfg)

    path = (
        OUT / f"fit-{key}.parquet"
    )

    print(
        f"[FIT {i:02d}/{len(BOUNDARY_CONFIGS)}]",
        key,
    )

    if path.is_file():
        df = pd.read_parquet(path)
        print("  CACHE HIT")
    else:
        df = run_pair_trajectory(
            cfg,
            fit_mask,
        )

        df.to_parquet(
            path,
            index=False,
        )

    fit_frames.append(df)

fit_boundary = pd.concat(
    fit_frames,
    ignore_index=True,
)

fit_slopes = slopes_from_trajectory(
    fit_boundary
)

assert (
    fit_slopes["query_id"].nunique()
    == int(fit_mask.sum())
)

assert (
    fit_slopes["config_key"].nunique()
    == 44
)

fit_slopes.to_parquet(
    OUT / "v013_fit_query_config_slopes.parquet",
    index=False,
)

print("FIT SLOPES:", fit_slopes.shape)
print("FEVER FIT SWEEP — COMPLETE")

## 9. Fit-only first-order boundary models

In [ ]:
fit_model_df = fit_slopes.merge(
    features_df,
    on="query_id",
    how="left",
    validate="many_to_one",
)

fit_model_df["is_softmax"] = (
    fit_model_df["method"]
    == "softmax"
).astype(float)

fit_model_df["temperature_numeric"] = (
    pd.to_numeric(
        fit_model_df["temperature"],
        errors="coerce",
    )
    .fillna(1.0)
)

FEATURES = [
    "initial_candidate_divergence",
    "initial_abs_utility_gap",
    "pq32_entropy20",
    "sq8_entropy20",
    "pq32_margin1_10",
    "sq8_margin1_10",
    "entropy_gap",
    "margin_gap",
    "alpha",
    "k",
    "is_softmax",
    "temperature_numeric",
]

TARGET = "H3_slope"

X_fit = (
    fit_model_df[FEATURES]
    .to_numpy(np.float64)
)

y_fit = (
    fit_model_df[TARGET]
    .to_numpy(np.float64)
)

ridge = Pipeline([
    (
        "scale",
        StandardScaler(),
    ),
    (
        "model",
        Ridge(alpha=10.0),
    ),
])

ridge.fit(
    X_fit,
    y_fit,
)

pred_fit = ridge.predict(X_fit)

high_threshold = float(
    np.quantile(
        y_fit,
        0.75,
    )
)

y_fit_high = (
    y_fit >= high_threshold
).astype(int)

clf = Pipeline([
    (
        "scale",
        StandardScaler(),
    ),
    (
        "model",
        LogisticRegression(
            C=0.5,
            max_iter=2000,
            class_weight="balanced",
        ),
    ),
])

clf.fit(
    X_fit,
    y_fit_high,
)

fit_probability = (
    clf.predict_proba(X_fit)[:, 1]
)

print(
    "FIT R2:",
    r2_score(
        y_fit,
        pred_fit,
    ),
)

print(
    "FIT MAE:",
    mean_absolute_error(
        y_fit,
        pred_fit,
    ),
)

print(
    "FIT AUC:",
    roc_auc_score(
        y_fit_high,
        fit_probability,
    ),
)

print(
    "Frozen FEVER high-amplification threshold:",
    high_threshold,
)

## 10. Held-out FEVER validation sweep

In [ ]:
val_frames = []

for i, cfg in enumerate(
    BOUNDARY_CONFIGS,
    1,
):
    key = cfg_key(cfg)

    path = (
        OUT / f"validation-{key}.parquet"
    )

    print(
        f"[VAL {i:02d}/{len(BOUNDARY_CONFIGS)}]",
        key,
    )

    if path.is_file():
        df = pd.read_parquet(path)
        print("  CACHE HIT")
    else:
        df = run_pair_trajectory(
            cfg,
            val_mask,
        )

        df.to_parquet(
            path,
            index=False,
        )

    val_frames.append(df)

val_boundary = pd.concat(
    val_frames,
    ignore_index=True,
)

val_slopes = slopes_from_trajectory(
    val_boundary
)

val_model_df = val_slopes.merge(
    features_df,
    on="query_id",
    how="left",
    validate="many_to_one",
)

val_model_df["is_softmax"] = (
    val_model_df["method"]
    == "softmax"
).astype(float)

val_model_df["temperature_numeric"] = (
    pd.to_numeric(
        val_model_df["temperature"],
        errors="coerce",
    )
    .fillna(1.0)
)

X_val = (
    val_model_df[FEATURES]
    .to_numpy(np.float64)
)

y_val = (
    val_model_df[TARGET]
    .to_numpy(np.float64)
)

pred_val = ridge.predict(X_val)

y_val_high = (
    y_val >= high_threshold
).astype(int)

p_val = (
    clf.predict_proba(X_val)[:, 1]
)

yhat_val = (
    p_val >= 0.5
).astype(int)

validation_metrics = {
    "r2":
        float(
            r2_score(
                y_val,
                pred_val,
            )
        ),
    "mae":
        float(
            mean_absolute_error(
                y_val,
                pred_val,
            )
        ),
    "auc":
        float(
            roc_auc_score(
                y_val_high,
                p_val,
            )
        ),
    "average_precision":
        float(
            average_precision_score(
                y_val_high,
                p_val,
            )
        ),
    "accuracy_at_0p5":
        float(
            accuracy_score(
                y_val_high,
                yhat_val,
            )
        ),
    "fit_high_amplification_threshold":
        high_threshold,
    "fit_high_prevalence":
        float(
            y_fit_high.mean()
        ),
    "validation_high_prevalence":
        float(
            y_val_high.mean()
        ),
}

print(
    json.dumps(
        validation_metrics,
        indent=2,
    )
)

print(
    "FEVER OUT-OF-SAMPLE BOUNDARY VALIDATION — COMPLETE"
)

## 11. Regime map and cross-dataset comparison

In [ ]:
EPS = 0.002

def regime(x):
    if x > EPS:
        return "amplifying"
    if x < -EPS:
        return "reversal"
    return "stable_or_null"

val_model_df["regime"] = [
    regime(x)
    for x in val_model_df["H3_slope"]
]

regime_counts = (
    val_model_df
    .groupby(
        "regime"
    )
    .agg(
        rows=("query_id", "size"),
        queries=("query_id", "nunique"),
        mean_H3_slope=("H3_slope", "mean"),
        mean_predicted_risk=(
            "H3_slope",
            lambda x: np.nan,
        ),
    )
    .reset_index()
)

risk_df = val_model_df[
    ["query_id", "config_key", "regime"]
].copy()

risk_df["predicted_risk"] = p_val

regime_risk = (
    risk_df
    .groupby(
        "regime",
        as_index=False,
    )
    .agg(
        rows=("query_id", "size"),
        queries=("query_id", "nunique"),
        mean_predicted_risk=(
            "predicted_risk",
            "mean",
        ),
    )
)

display(regime_risk)

regime_risk.to_csv(
    OUT / "v013_validation_regime_risk.csv",
    index=False,
)

# HotpotQA v0.9 is read only after FEVER validation is complete.
hotpot_runs = sorted(
    [
        p
        for p in HOTPOT_V09_ROOT.glob("*")
        if (p / "report.json").is_file()
    ],
    key=lambda p: p.stat().st_mtime,
)

cross_dataset = {
    "fever_validation_auc":
        validation_metrics["auc"],
    "fever_validation_ap":
        validation_metrics["average_precision"],
    "fever_validation_r2":
        validation_metrics["r2"],
}

if hotpot_runs:
    with open(
        hotpot_runs[-1] / "report.json",
        "r",
        encoding="utf-8",
    ) as f:
        hotpot_report = json.load(f)

    cross_dataset.update({
        "hotpotqa_validation_auc":
            float(
                hotpot_report[
                    "validation_metrics"
                ]["auc"]
            ),
        "hotpotqa_validation_ap":
            float(
                hotpot_report[
                    "validation_metrics"
                ]["average_precision"]
            ),
        "hotpotqa_validation_r2":
            float(
                hotpot_report[
                    "validation_metrics"
                ]["r2"]
            ),
    })

display(
    pd.DataFrame(
        [cross_dataset]
    )
)

## 12. Coefficients and replication figures

In [ ]:
coef_df = pd.DataFrame({
    "feature": FEATURES,
    "ridge_standardized_coefficient":
        ridge.named_steps["model"].coef_,
    "logistic_standardized_coefficient":
        clf.named_steps["model"].coef_[0],
})

coef_df.to_csv(
    OUT / "v013_boundary_model_coefficients.csv",
    index=False,
)

display(
    coef_df.reindex(
        coef_df[
            "logistic_standardized_coefficient"
        ]
        .abs()
        .sort_values(
            ascending=False
        )
        .index
    )
)

fig, ax = plt.subplots(
    figsize=(6.5, 4.5)
)

ax.bar(
    ["FEVER"],
    [validation_metrics["auc"]],
)

if (
    "hotpotqa_validation_auc"
    in cross_dataset
):
    ax.bar(
        ["HotpotQA v0.9"],
        [
            cross_dataset[
                "hotpotqa_validation_auc"
            ]
        ],
    )

ax.axhline(
    0.5,
    linewidth=1,
)

ax.set_ylim(
    0.45,
    1.0,
)

ax.set_ylabel(
    "Held-out ROC-AUC"
)

ax.set_title(
    "Boundary Prediction Across Datasets"
)

fig.tight_layout()

FIG = (
    OUT
    / "v013_cross_dataset_boundary_auc.png"
)

fig.savefig(
    FIG,
    dpi=180,
    bbox_inches="tight",
)

plt.show()

## 13. Seal ARC-v0.13 report

In [ ]:
val_model_df.to_parquet(
    OUT / "v013_boundary_validation_query_config_rows.parquet",
    index=False,
)

replication_decision = {
    "held_out_auc_above_random":
        bool(
            validation_metrics["auc"]
            > 0.5
        ),
    "held_out_ap_above_prevalence":
        bool(
            validation_metrics[
                "average_precision"
            ]
            >
            validation_metrics[
                "validation_high_prevalence"
            ]
        ),
    "null_and_reversal_retained":
        bool(
            set(
                val_model_df[
                    "regime"
                ].unique()
            )
            >= {
                "amplifying",
                "stable_or_null",
                "reversal",
            }
        ),
}

report = {
    "status":
        "ARC_V013_FEVER_BOUNDARY_EXTERNAL_REPLICATION_COMPLETE",
    "protocol_sha256":
        protocol_sha,
    "dataset":
        "FEVER",
    "dev_queries":
        len(dev_ids),
    "fit_queries":
        int(fit_mask.sum()),
    "validation_queries":
        int(val_mask.sum()),
    "boundary_grid_config_count":
        len(BOUNDARY_CONFIGS),
    "prediction_target":
        TARGET,
    "features":
        FEATURES,
    "validation_metrics":
        validation_metrics,
    "regime_threshold_abs_slope":
        EPS,
    "replication_decision":
        replication_decision,
    "cross_dataset_reference":
        cross_dataset,
    "retain_null_and_reversal_cases":
        True,
    "test_accessed":
        False,
    "completed_at_utc":
        datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = (
    OUT / "report.json"
)

REPORT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        default=float,
    ),
    encoding="utf-8",
)

report_sha = sha256_file(
    REPORT_PATH
)

(
    OUT / "REPORT_SHA256.txt"
).write_text(
    report_sha + "\n",
    encoding="utf-8",
)

print()
print("=" * 80)
print(
    "ARC-v0.13 FEVER BOUNDARY EXTERNAL REPLICATION — COMPLETE"
)
print("Output:", OUT)
print("Report SHA-256:", report_sha)
print("=" * 80)
print(
    json.dumps(
        report,
        indent=2,
        default=float,
    )
)

## Interpretation discipline

Do not classify the replication solely from whether the FEVER AUC numerically matches HotpotQA.

The paper-facing interpretation must distinguish:

- **phenomenon replication** — whether FEVER retains heterogeneous amplification, stable/null, and reversal regimes;
- **predictive replication** — whether the frozen first-order feature family has held-out discrimination above chance;
- **effect-size transport** — whether discrimination and slopes are similar or materially weaker/stronger than HotpotQA;
- **external-validity limits** — this is still the same BGE-small encoder and PQ32↔SQ8 approximation family.

A weaker or null FEVER boundary predictor is scientifically useful and must be retained.